# Predator-prey: births and deaths every step

The bundled Wolf-Sheep model, scaled. Every step moves every animal, kills the starving and spawns offspring, so this is the agent-lifecycle workload: the cost per agent-step includes a share of births and deaths.

In [1]:
import json
import time

import pandas as pd


def bench(label, make_model, steps, agents_of=lambda m: len(m.agents)):
    """Build the model, time ``steps`` steps, print a RESULT line, return a row."""
    model = make_model()
    n = agents_of(model)
    t0 = time.perf_counter()
    model.run_for(steps)
    per_step = (time.perf_counter() - t0) / steps
    row = {
        "label": label,
        "agents": n,
        "ms_per_step": round(per_step * 1e3, 2),
        "us_per_agent_step": round(per_step / max(n, 1) * 1e6, 2),
    }
    print("RESULT " + json.dumps(row))
    return row

In [2]:
from modelly.examples.advanced.wolf_sheep.model import WolfSheep, WolfSheepScenario


def wolf_sheep(side, sheep, wolves):
    return lambda: WolfSheep(
        WolfSheepScenario(height=side, width=side, initial_sheep=sheep, initial_wolves=wolves,
                          sheep_reproduce=0.4, wolf_reproduce=0.2, grass_regrowth_time=10, rng=1)
    )

In [3]:
rows = []
rows.append(bench("50x50, 250 sheep, 125 wolves", wolf_sheep(50, 250, 125), steps=20))
rows.append(bench("100x100, 1000 sheep, 500 wolves", wolf_sheep(100, 1000, 500), steps=20))
rows.append(bench("200x200, 4000 sheep, 2000 wolves", wolf_sheep(200, 4000, 2000), steps=10))

RESULT {"label": "50x50, 250 sheep, 125 wolves", "agents": 2875, "ms_per_step": 4.02, "us_per_agent_step": 1.4}


RESULT {"label": "100x100, 1000 sheep, 500 wolves", "agents": 11500, "ms_per_step": 15.12, "us_per_agent_step": 1.31}


RESULT {"label": "200x200, 4000 sheep, 2000 wolves", "agents": 46000, "ms_per_step": 79.52, "us_per_agent_step": 1.73}


In [4]:
pd.DataFrame(rows).set_index('label')

,agents,ms_per_step,us_per_agent_step
label,,,
"50x50, 250 sheep, 125 wolves",2875,4.02,1.40
"100x100, 1000 sheep, 500 wolves",11500,15.12,1.31
"200x200, 4000 sheep, 2000 wolves",46000,79.52,1.73
